# EPF (final)
By now, we strive on creatng an EPF that generates a final dataset with usable features for the cretioon of our final dataset for RL bank management model.

---
### Standardizing our workflow
By now, we're gonna work out too in standardize the process of **creating datesets for the RL bank management model**. 

It may look like this:
> 1. Importing the method ```setup()```
> 2. Importing main libraries
> 3. Importing dataset
> 4. Applying the ```setup()``` pre-processment method

Until here, things are gonna be quite similar in all our final versions of risk envelope tools. The ```setup()``` method take care of a lot of the original datasets problems including:

- Flipping "Date" feature order (from decrescent to crescent)
- Spliting tests and train

Then, we keep going:
> 5. Creating the model
> 6. Testing it (sanity tests) ***NOT DONE HERE!***
> 7. Creation of features for RL bank management dataset
> 8. Saving the dataset



# 1. Importing the method ```setup()```

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("../../src").resolve()))
from setup import setup

## 2. Importing main libraries

In [2]:
from setup import setup
import pandas as pd
import numpy as np

## 3. Importing dataset

In [3]:
df = pd.read_csv("../../data/processed/financial_tools_datset.csv")


## 4. Applying the ```setup()``` pre-processment method

In [4]:
splits = setup(
    csv_path="../../data/processed/financial_tools_datset.csv",
    target_col="Price",
    horizon=1
)

### 4.1 Splits printing
The splits were created in ```setup()``` method application

In [5]:
print("Train shape:", splits["X_train"].shape)
print("Val shape:", splits["X_val"].shape)
print("Test shape:", splits["X_test"].shape)

print("Número de features:", len(splits["feature_columns"]))

Train shape: (956, 24)
Val shape: (205, 24)
Test shape: (206, 24)
Número de features: 24


## 5. EPF model creation

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# Target binário
y_train_bin = (splits["y_train"] > 0).astype(int)
y_val_bin = (splits["y_val"] > 0).astype(int)

model = LogisticRegression(max_iter=1000)

model.fit(splits["X_train"], y_train_bin)

val_probs = model.predict_proba(splits["X_val"])[:, 1]

auc = roc_auc_score(y_val_bin, val_probs)

print("Validation AUC:", auc)

Validation AUC: 0.9307062630877593


In [7]:
epf_probs = pd.Series(val_probs)

epf_score = epf_probs.rolling(50).mean()

epf_score.name = "epf_score"
epf_score

0           NaN
1           NaN
2           NaN
3           NaN
4           NaN
         ...   
200    0.514912
201    0.521102
202    0.504650
203    0.484960
204    0.471319
Name: epf_score, Length: 205, dtype: float64

**Right here is where the sanity test have to be displayed! Since this notebook is not engaged in test the reliability of the model (we already have tested it in ```/notebooks/risk_envelop/epf.ipynb```), we ignore this part by now and keep working on creating our features for the new RL bank management dataset model**

## Creation of features for RL bank management dataset
We use the EPF score as basi for the rest of the data features

In [8]:
price = df["Price"].iloc[-len(epf_score):].reset_index(drop=True)

epf_upper = price * (1 + epf_score)
epf_lower = price * (1 - epf_score)

In [9]:
epf_dataset = pd.DataFrame()

epf_dataset["epf_upper"] = epf_upper
epf_dataset["epf_lower"] = epf_lower

In [10]:
epf_dataset["epf_width"] = epf_dataset["epf_upper"] - epf_dataset["epf_lower"]

In [11]:
mid = (epf_dataset["epf_upper"] + epf_dataset["epf_lower"]) / 2

epf_dataset["epf_asymmetry"] = (price - mid) / epf_dataset["epf_width"]

In [12]:
epf_dataset.tail()

,epf_upper,epf_lower,epf_width,epf_asymmetry
200,1.641255,0.525545,1.115711,0.0
201,1.647353,0.518647,1.128707,0.0
202,1.631040,0.536960,1.094081,0.0
203,1.614299,0.559901,1.054399,0.0
204,1.605797,0.577003,1.028795,0.0


## Solving the size of dataset problem
The dataset generated by our ```epf_dataset``` is only 205 rows. We'll need far more data **to synchornize** with the data that will be generated by the other risk envelope tools. So we'll use this dataset expansion trick to get this dataset to 1568 rows (the size of the original dataset).

In [13]:
epf_full = pd.DataFrame(index=df.index, columns=epf_dataset.columns)

epf_full.iloc[-len(epf_dataset):] = epf_dataset.values

epf_full = epf_full.ffill()

epf_full

C:\Users\Miguel\AppData\Local\Temp\ipykernel_9292\1941782656.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  epf_full = epf_full.ffill()


,epf_upper,epf_lower,epf_width,epf_asymmetry
0,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN
...,...,...,...,...
1563,1.641255,0.525545,1.115711,0.0
1564,1.647353,0.518647,1.128707,0.0
1565,1.631040,0.536960,1.094081,0.0
1566,1.614299,0.559901,1.054399,0.0


In [14]:
epf_full.shape

(1568, 4)

### Fim